# 1. Coneccion con Amazon

En Amazon, tienen un sistema anti codigo, osea que si intento bajar informacion me saldra estado de codigo 503 (Servicio no disponible), por eso gracias a una linea de codigo headers se puede llegar al esta 202 ( solcitud aceptada semi completa)

In [ ]:
import requests

url = "https://www.amazon.com"

respuesta = requests.get(url)

print("Status code:", respuesta.status_code)

if respuesta.status_code == 200:
    print("Conexión exitosa")
else:
    print("No se pudo acceder correctamente")

Status code: 503
No se pudo acceder correctamente


In [ ]:
import requests

url = "https://www.amazon.com"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0 Safari/537.36"
}

respuesta = requests.get(url, headers=headers)

print("Status code:", respuesta.status_code)
print("Content-Type:", respuesta.headers.get("Content-Type"))

Status code: 202
Content-Type: text/html; charset=UTF-8


producto = "audifinos inalambricos"

https://www.amazon.com/s?k=caja&__mk_es_US=%C3%85M%C3%85%C5%BD%C3%95%C3%91&crid=2EJ28KPN713RD&sprefix=ca%2Caps%2C356&ref=nb_sb_noss_2

https://www.amazon.com/s?k=audifonos+inalambricos&crid=22KU3N8QD3H7J&sprefix=aud%2Caps%2C368&ref=nb_sb_ss_p13n-expert-pd-ops-ranker_1_3

Dado que el producto que busco es audifonos inalambricos, lo pongo en el buscador de la pagina y copio el http, al comparar 2 busquedas me doy cuenta lo que cambia, y es que los espacios en blanco lo remplaza con un "+".

Pero las letras siguientes a lo que queremos buscar no son nesesarios, si solo usamos el:

https://www.amazon.com/s?k=audifonos+inalambricos

Tambien funcionara

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

producto = "audifonos inalambricos"
url = f"https://www.amazon.com/s?k={producto.replace(' ', '+')}"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
}

print("---------------------------------")

respuesta = requests.get(url, headers=headers)
soup = BeautifulSoup(respuesta.text, "html.parser")
productos = soup.select("div[data-component-type='s-search-result']")
listas = []
for i, item in enumerate(productos[:5], start=1):
    titulo = item.select_one("h2 span")
    id = item.get("data-asin")
    if titulo:
        print(f"{i}. {titulo.get_text(strip=True)} (ID: {id})")
        listas.append((titulo.get_text(strip=True), id))

with open("productos.csv", "w", newline="", encoding="utf-8") as archivo:
    escritor = csv.writer(archivo)
    escritor.writerow(["nombre_producto", "id_producto"])
    escritor.writerows(listas)

---------------------------------
1. Soundcore P30i by Anker Noise Cancelling Earbuds, Strong and Smart Noise Cancelling, Powerful Bass, 45H Playtime, 2-in-1 Case and Phone Stand, IP54, Wireless Earbuds, Bluetooth 5.4 (Green) (ID: B0CRTR3PMF)
2. Wireless Earbuds Bluetooth 5.4 Headphones, 54H Playtime Stereo Earphones with Noise Cancelling Mic, IPX7 Waterproof, Fast Charging and Comfort Fit Ear Buds for Fitness/Sports/Workouts/Travel Black (ID: B0H1WSG5J7)
3. Wireless Earbuds, Bluetooth 5.3 Headphones HiFi Stereo 50H Playback LED Digital Display Ear Buds with ENC Noise Canceling Headset, IPX7 Waterproof Earphones for Gym/Running/Work (White) (ID: B0H2VWNKN6)
4. Soundcore by Anker P20i True Wireless Earbuds, 10mm Drivers with Big Bass, Bluetooth 5.3, 30H Long Playtime, Water-Resistant, 2 Mics for AI Clear Calls, 22 Preset EQs, Customization via App (ID: B0BTYCRJSS)
5. PocBuds Bluetooth Headphones Wireless Earbuds 80hrs Playtime Wireless Charging Case Digital Display Sports Ear Buds with 

La idea es que cada audifono que me extraiga el codigo anterior me salgan los comentarios, los cuales poseen esta estructura:

pagina de reseñas articulo 1:

https://www.amazon.com/-/es/Auriculares-inal%C3%A1mbricos-auriculares-cancelaci%C3%B3n-entrenamientos/dp/B0H1WSG5J7/ref=sr_1_3?dib=eyJ2IjoiMSJ9.Jold3rIibnbgvuXHxOIJc3fy9jxiU74iyxr8BFLLMwV9V7o3TxYHZmsBYnG-rAfQa0sb2VUuG88gPHTOs8E64z6PtaCoh3Kq5S2PMI19BlyuPACGenBUsNlRU5n1vM6HmxrlsojLZRjvBIQ-i6Shx5odBA0BLY9ySQtblnxQOK8TAOuXJXiFAokBlerZxeX1dy2n8ppFbBbTUUUxx3wWKm3TknnqzRakEadfYhvkJxw.EPWFEvDYJRBfU3cLcvOjtGTGERTguD_oYFTocXIjOpw&dib_tag=se&keywords=audifonos%2Binalambricos&qid=1782062691&sr=8-3&th=1


Pagina de reseñas articulo 1 reducido:
https://www.amazon.com/-/es/dp/B0H1WSG5J7/

![imagen](html_inspeccionar.png)

Para cada producto, se le asigna un id unico que se ubica en "data-asin", para el ejemplo, en ves de usar todo el link, se reduce y nos facilita la busqueda.
...

NOTA: En esta parte voy a analizar el codigo html para extraer los comentarios de audifono inalambrico con id B0H1WSG5J7, dado que la pagina usa este modelo de escritura para la mayoria, me servira de plantilla para los productos

In [ ]:
import requests

url = "https://www.amazon.com/-/es/dp/B0H1WSG5J7/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8"
}

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)

html = response.text

Status code: 200


Nota: codigo qie imprime la informacion pero no los comentarios, ya que amazon usa comentarios dimanicos.

In [ ]:
import requests
import csv

ASIN = "B0H1WSG5J7"

url = f"https://www.amazon.com/-/es/dp/{ASIN}/"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

reviews = soup.find_all("div", {"data-hook": "review"})

data = []

for r in reviews:

    # comentario
    comentario = r.find("span", {"class":"_Y3Itd_contain-rich-content_2IORW"})
    comentario = comentario.text.strip() if comentario else ""

    # usuario
    usuario = r.find("span", {"class": "a-profile-name"})
    usuario = usuario.text.strip() if usuario else ""

    # estrellas
    estrellas = r.find("i", {"data-hook": "review-star-rating"})
    if estrellas:
        estrellas = estrellas.text.strip()
    else:
        estrellas = ""

    # fecha
    fecha = r.find("span", {"data-hook": "review-date"})
    fecha = fecha.text.strip() if fecha else ""

    data.append([comentario, usuario, estrellas, fecha])

print(data)

[['', 'Yusbeidy Pérez', '5 de 5 estrellas', 'Calificado en Estados Unidos el 18 de junio de 2026'], ['', 'Geysa Ramirez', '5 de 5 estrellas', 'Calificado en Estados Unidos el 18 de junio de 2026'], ['', 'María Isabel Monasterio Gamez', '5 de 5 estrellas', 'Calificado en Estados Unidos el 19 de junio de 2026'], ['', 'Your favorite finds with Ailen', '5 de 5 estrellas', 'Calificado en Estados Unidos el 21 de junio de 2026'], ['', 'Amazon Customer', '4 de 5 estrellas', 'Calificado en Estados Unidos el 21 de junio de 2026'], ['', 'Gabriel W.', '3 de 5 estrellas', 'Calificado en Estados Unidos el 19 de junio de 2026'], ['', 'Yaneisy Lohuis', '5 de 5 estrellas', 'Calificado en Estados Unidos el 18 de junio de 2026'], ['', 'Leydi Corvo Fernández', '5 de 5 estrellas', 'Calificado en Estados Unidos el 18 de junio de 2026'], ['', 'Laura Wingate', '5 de 5 estrellas', 'Calificado en Reino Unido el 1 de mayo de 2026'], ['', 'K McNeil', '5 de 5 estrellas', 'Calificado en Reino Unido el 5 de abril de

Codigo para segurarme que en el html esta los comentarios de la pagina web.

In [ ]:
from bs4 import BeautifulSoup
import csv
import re

# Leer el archivo HTML guardado
with open("amazom.txt", "r", encoding="utf-8") as file:
    html_content = file.read()

soup = BeautifulSoup(html_content, "html.parser")

# Buscar reseñas usando múltiples estrategias
review_elements = soup.find_all("div", {"data-hook": "review"})

# Si no encuentra, buscar por clases que contengan "review"
if not review_elements:
    review_elements = soup.find_all("div", class_=re.compile(r"review"))

# Si aún no encuentra, buscar en la sección de reseñas
if not review_elements:
    review_section = soup.find("div", {"id": "customerReviews"})
    if review_section:
        review_elements = review_section.find_all("div", class_=re.compile(r"review"))

# Textos a eliminar
textos_a_eliminar = [
    "Brief content visible, double tap to read full content.",
    "Full content visible, double tap to read brief content.",
    "Brief content visible, double tap to read full content",
    "Full content visible, double tap to read brief content",
]

def limpiar_comentario(texto):
    """Limpia el comentario eliminando los textos no deseados"""
    if not texto:
        return ""

    # Eliminar cada texto no deseado
    for texto_eliminar in textos_a_eliminar:
        texto = texto.replace(texto_eliminar, "")

    # Eliminar espacios extra al inicio y final
    texto = texto.strip()

    # Eliminar saltos de línea múltiples
    texto = re.sub(r'\n\s*\n', '\n', texto)

    return texto

data = []

for r in review_elements:
    try:
        # COMERNTARIO - Buscar con múltiples selectores
        comentario = None

        # Intentar con data-hook
        comentario_elem = r.find("span", {"data-hook": "review-body"})
        if comentario_elem:
            comentario = limpiar_comentario(comentario_elem.text.strip())

        # Si no, buscar en div con clase review-text
        if not comentario:
            comentario_elem = r.find("div", {"data-hook": "review-collapsed"})
            if comentario_elem:
                comentario = limpiar_comentario(comentario_elem.text.strip())

        # Si no, buscar cualquier div que contenga "review-text"
        if not comentario:
            comentario_elem = r.find("div", class_=re.compile(r"review-text"))
            if comentario_elem:
                comentario = limpiar_comentario(comentario_elem.text.strip())

        # Si no, buscar en el contenido rich
        if not comentario:
            comentario_elem = r.find("div", {"data-hook": "reviewRichContentContainer"})
            if comentario_elem:
                comentario = limpiar_comentario(comentario_elem.text.strip())

        # Si no, buscar párrafos dentro de la reseña
        if not comentario:
            p_elem = r.find("p")
            if p_elem:
                comentario = limpiar_comentario(p_elem.text.strip())

        # USUARIO
        usuario = None
        usuario_elem = r.find("span", {"class": "a-profile-name"})
        if usuario_elem:
            usuario = usuario_elem.text.strip()

        # Si no, buscar en el perfil
        if not usuario:
            profile_elem = r.find("a", {"class": "a-profile"})
            if profile_elem:
                profile_name = profile_elem.find("span", {"class": "a-profile-name"})
                if profile_name:
                    usuario = profile_name.text.strip()

        # ESTRELLAS
        estrellas = None
        estrellas_elem = r.find("i", {"data-hook": "review-star-rating"})
        if estrellas_elem:
            estrellas = estrellas_elem.text.strip()

        # Si no, buscar cualquier estrella
        if not estrellas:
            estrellas_elem = r.find("i", class_=re.compile(r"a-icon-star"))
            if estrellas_elem:
                estrellas = estrellas_elem.text.strip()

        # FECHA
        fecha = None
        fecha_elem = r.find("span", {"data-hook": "review-date"})
        if fecha_elem:
            fecha = fecha_elem.text.strip()

        # Si no, buscar fecha en otra clase
        if not fecha:
            fecha_elem = r.find("span", class_=re.compile(r"review-date"))
            if fecha_elem:
                fecha = fecha_elem.text.strip()

        # TÍTULO DE LA RESEÑA
        titulo = None
        titulo_elem = r.find("a", {"data-hook": "review-title"})
        if titulo_elem:
            titulo = titulo_elem.text.strip()

        if not titulo:
            titulo_elem = r.find("h5", {"data-hook": "reviewTitle"})
            if titulo_elem:
                titulo = titulo_elem.text.strip()

        # Solo agregar si al menos tiene comentario o usuario
        if comentario or usuario:
            data.append({
                "comentario": comentario or "No disponible",
                "usuario": usuario or "Anónimo",
                "estrellas": estrellas or "No especificado",
                "fecha": fecha or "No especificada",
                "titulo": titulo or "Sin título"
            })

    except Exception as e:
        print(f"Error procesando reseña: {e}")
        continue

print(f"Se encontraron {len(data)} reseñas")

# Guardar en CSV
with open("reviews_extracted.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["comentario", "usuario", "estrellas", "fecha", "titulo"])
    writer.writeheader()
    writer.writerows(data)

# Mostrar primeras reseñas como preview
print("\n=== PRIMERAS RESEÑAS ===\n")
for i, review in enumerate(data, 1):
    print(f"Reseña {i}:")
    print(f"Usuario: {review['usuario']}")
    print(f"Estrellas: {review['estrellas']}")
    print(f"Fecha: {review['fecha']}")
    print(f"Título: {review['titulo']}")
    print(f"Comentario: {review['comentario'][:200]}...")
    print("-" * 50)

Se encontraron 13 reseñas

=== PRIMERAS RESEÑAS ===

Reseña 1:
Usuario: Yusbeidy Pérez
Estrellas: 5 de 5 estrellas
Fecha: Calificado en Estados Unidos el 18 de junio de 2026
Título: Safe and good noise cancelación
Comentario: I bought these Bluetooth 5.4 headphones with stereo sound and 54 hours of battery life. They include noise-canceling microphones, have a secure and comfortable fit, and come with IPX7 certification (r...
--------------------------------------------------
Reseña 2:
Usuario: Geysa Ramirez
Estrellas: 5 de 5 estrellas
Fecha: Calificado en Estados Unidos el 18 de junio de 2026
Título: Wireless Earbuds
Comentario: I like this style super comfortable for the ears and it sounds good you don't feel like you're wearing anything in your ears I liked themLeer másLeer menos...
--------------------------------------------------
Reseña 3:
Usuario: María Isabel Monasterio Gamez
Estrellas: 5 de 5 estrellas
Fecha: Calificado en Estados Unidos el 19 de junio de 2026
Título: I love
C

REUNION DE IDEAS:

Dado que cada "audifono inalambrico" posee un ID unico, y con el sitio web https://www.amazon.com/-/es/dp/{ID}/, asi que hay 2 formas, hacerlo uno por uno o automatizarlo y colocar un solo csv donde esten todos los comentarios disponibles con el codigo ID unico como nueva columna, ya que poner el nombre del producto es muy largo.

In [ ]:
import csv
import requests

with open("productos.csv", "r", encoding="utf-8") as archivo:
    lector = csv.DictReader(archivo)
    ids = [fila["id_producto"] for fila in lector]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8"
}

nom = []

for i, asin in enumerate(ids, start=1):
    url = f"https://www.amazon.com/-/es/dp/{asin}"
    response = requests.get(url, headers=headers)
    html = response.text
    nombre_archivo = f"producto_{i}_{asin}.txt"

    with open(nombre_archivo, "w", encoding="utf-8") as archivo_txt:
        archivo_txt.write(html)

    nom.append(nombre_archivo)

print(nom)

def limpiar_comentario(texto):
        """Limpia el comentario eliminando los textos no deseados"""
        if not texto:
            return ""

        # Eliminar cada texto no deseado
        for texto_eliminar in textos_a_eliminar:
            texto = texto.replace(texto_eliminar, "")

        # Eliminar espacios extra al inicio y final
        texto = texto.strip()

        # Eliminar saltos de línea múltiples
        texto = re.sub(r'\n\s*\n', '\n', texto)

        return texto

['producto_1_B0CRTR3PMF.txt', 'producto_2_B0H1WSG5J7.txt', 'producto_3_B0H2VWNKN6.txt', 'producto_4_B0BTYCRJSS.txt', 'producto_5_B0C3W4MNN1.txt']


In [ ]:
for i in range(len(nom)):
    nombre_archivo = nom[i]
    print(nombre_archivo)

producto_1_B0H1WSG5J7.txt
producto_2_B0CRTR3PMF.txt
producto_3_B0H2VWNKN6.txt
producto_4_B0BTYCRJSS.txt
producto_5_B0C3W4MNN1.txt


In [ ]:
from bs4 import BeautifulSoup
import csv
import re

data = []

for i in range(len(nom)):
    nombre_archivo = nom[i]
    with open(nombre_archivo, "r", encoding="utf-8") as file:
        html_content = file.read()

    soup = BeautifulSoup(html_content, "html.parser")

    # Buscar reseñas usando múltiples estrategias
    review_elements = soup.find_all("div", {"data-hook": "review"})

    # Si no encuentra, buscar por clases que contengan "review"
    if not review_elements:
        review_elements = soup.find_all("div", class_=re.compile(r"review"))

    # Si aún no encuentra, buscar en la sección de reseñas
    if not review_elements:
        review_section = soup.find("div", {"id": "customerReviews"})
        if review_section:
            review_elements = review_section.find_all("div", class_=re.compile(r"review"))

    # Textos a eliminar
    textos_a_eliminar = [
        "Brief content visible, double tap to read full content.",
        "Full content visible, double tap to read brief content.",
        "Brief content visible, double tap to read full content",
        "Full content visible, double tap to read brief content",
    ]



    for r in review_elements:
        try:
            # COMERNTARIO - Buscar con múltiples selectores
            comentario = None

            # Intentar con data-hook
            comentario_elem = r.find("span", {"data-hook": "review-body"})
            if comentario_elem:
                comentario = limpiar_comentario(comentario_elem.text.strip())

            # Si no, buscar en div con clase review-text
            if not comentario:
                comentario_elem = r.find("div", {"data-hook": "review-collapsed"})
                if comentario_elem:
                    comentario = limpiar_comentario(comentario_elem.text.strip())

            # Si no, buscar cualquier div que contenga "review-text"
            if not comentario:
                comentario_elem = r.find("div", class_=re.compile(r"review-text"))
                if comentario_elem:
                    comentario = limpiar_comentario(comentario_elem.text.strip())

            # Si no, buscar en el contenido rich
            if not comentario:
                comentario_elem = r.find("div", {"data-hook": "reviewRichContentContainer"})
                if comentario_elem:
                    comentario = limpiar_comentario(comentario_elem.text.strip())

            # Si no, buscar párrafos dentro de la reseña
            if not comentario:
                p_elem = r.find("p")
                if p_elem:
                    comentario = limpiar_comentario(p_elem.text.strip())

            # USUARIO
            usuario = None
            usuario_elem = r.find("span", {"class": "a-profile-name"})
            if usuario_elem:
                usuario = usuario_elem.text.strip()

            # Si no, buscar en el perfil
            if not usuario:
                profile_elem = r.find("a", {"class": "a-profile"})
                if profile_elem:
                    profile_name = profile_elem.find("span", {"class": "a-profile-name"})
                    if profile_name:
                        usuario = profile_name.text.strip()

            # ESTRELLAS
            estrellas = None
            estrellas_elem = r.find("i", {"data-hook": "review-star-rating"})
            if estrellas_elem:
                estrellas = estrellas_elem.text.strip()


            # FECHA
            fecha = None
            fecha_elem = r.find("span", {"data-hook": "review-date"})
            if fecha_elem:
                fecha = fecha_elem.text.strip()

            # TÍTULO DE LA RESEÑA
            titulo = None
            titulo_elem = r.find("a", {"data-hook": "review-title"})
            if titulo_elem:
                titulo = titulo_elem.text.strip()

            if not titulo:
                titulo_elem = r.find("h5", {"data-hook": "reviewTitle"})
                if titulo_elem:
                    titulo = titulo_elem.text.strip()

            # Solo agregar si al menos tiene comentario o usuario
            if comentario or usuario:
                data.append({
                    "comentario": comentario or "No disponible",
                    "usuario": usuario or "Anónimo",
                    "estrellas": estrellas or "No especificado",
                    "fecha": fecha or "No especificada",
                    "titulo": titulo or "Sin título"
                })

        except Exception as e:
            print(f"Error procesando reseña: {e}")
            continue

    print(f"Se encontraron {len(data)} reseñas")

    # Guardar en CSV
    with open("reviews_extracted.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["comentario", "usuario", "estrellas", "fecha", "titulo"])
        writer.writeheader()
        writer.writerows(data)


Se encontraron 13 reseñas
Se encontraron 26 reseñas
Se encontraron 39 reseñas
Se encontraron 52 reseñas
Se encontraron 65 reseñas


In [ ]:
import pandas as pd

df = pd.read_csv("reviews_extracted.csv")

print(df.iloc[:, 3])

0     Calificado en Estados Unidos el 4 de junio de ...
1     Calificado en Estados Unidos el 5 de junio de ...
2     Calificado en Estados Unidos el 1 de enero de ...
3     Calificado en Estados Unidos el 23 de abril de...
4     Calificado en Estados Unidos el 20 de junio de...
                            ...                        
60    Calificado en Emiratos Árabes Unidos el 19 de ...
61    Calificado en Arabia Saudita el 9 de junio de ...
62          Calificado en México el 12 de abril de 2024
63         Calificado en México el 3 de octubre de 2023
64        Calificado en México el 30 de octubre de 2023
Name: fecha, Length: 65, dtype: object


# Análisis final de decisión de compra en Amazon

En esta sección se realiza el análisis de los productos obtenidos mediante scraping de Amazon para la búsqueda de **parlantes Bluetooth**.

El objetivo es analizar los primeros 100 productos considerando principalmente el precio, las estrellas, los comentarios y la comparación entre precio normal y precio promocional.

Para el análisis se utiliza el archivo CSV generado previamente por el scraper. Además, se aplican expresiones regulares para limpiar los datos numéricos, pandas para el procesamiento de datos y matplotlib para la elaboración de gráficos.

In [ ]:
# ============================================================
# ANÁLISIS FINAL - DECISIÓN DE COMPRA EN AMAZON
# Parlantes Bluetooth
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# Cargar archivo generado por el scraper
df = pd.read_csv("parlantes_bluetooth_amazon.csv")

print("Filas y columnas:", df.shape)
print("Columnas disponibles:", df.columns.tolist())

df.head()

## Limpieza de datos

El archivo CSV obtenido contiene datos en formato de texto, por ejemplo precios con símbolos o estrellas escritas como texto.

Por ese motivo, se utiliza una función con **expresiones regulares** para extraer únicamente los valores numéricos. Esto permite convertir los precios y las estrellas a datos numéricos para poder analizarlos con pandas.

In [ ]:
# ============================================================
# LIMPIEZA DE DATOS CON EXPRESIONES REGULARES
# ============================================================

def extraer_numero(texto):
    """
    Extrae el primer número encontrado dentro de un texto usando regex.

    Ejemplos:
    "$29.99" -> 29.99
    "4.5 out of 5 stars" -> 4.5
    "1,250 comentarios" -> 1250
    """

    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    # Patrón para encontrar números enteros o decimales
    patron = r"[0-9]+(?:,[0-9]{3})*(?:\.[0-9]+)?"

    encontrado = re.findall(patron, texto)

    if len(encontrado) == 0:
        return np.nan

    numero = encontrado[0].replace(",", "")

    return float(numero)


# Convertir la columna precio a número
df["precio_num"] = df["precio"].apply(extraer_numero)

# Convertir la columna estrellas a número
df["rating"] = df["estrellas"].apply(extraer_numero)

# Seleccionar los primeros 100 productos
df_100 = df.head(100).copy()

# Eliminar productos que no tengan precio válido
df_precio = df_100.dropna(subset=["precio_num"]).copy()

print("Total de productos analizados:", len(df_100))
print("Productos con precio válido:", len(df_precio))

df_precio.head()

## 1. Gráficos acerca del precio de los 100 primeros productos

Para analizar los precios de los productos se elaboran dos gráficos:

1. Un **gráfico de caja**, que permite observar la distribución general de los precios, la mediana y posibles valores atípicos.
2. Un **gráfico de barras**, que permite comparar visualmente los precios de los primeros productos con precio disponible.

In [ ]:
# ============================================================
# GRÁFICO DE CAJA DEL PRECIO
# ============================================================

plt.figure(figsize=(8, 5))
plt.boxplot(df_precio["precio_num"], vert=True)

plt.title("Distribución de precios de los parlantes Bluetooth")
plt.ylabel("Precio")
plt.grid(axis="y", alpha=0.3)

plt.show()

### Interpretación del gráfico de caja

El gráfico de caja muestra cómo se distribuyen los precios de los parlantes Bluetooth.  
Este gráfico permite identificar si la mayoría de productos se encuentra dentro de un rango similar de precios o si existen productos con precios muy altos o muy bajos en comparación con el resto.

In [ ]:
# ============================================================
# GRÁFICO DE BARRAS DE PRECIOS
# ============================================================

# Tomamos los primeros 20 productos con precio válido para que el gráfico sea legible
muestra = df_precio.head(20).copy()

# Crear nombres cortos para los productos
muestra["producto_n"] = ["Producto " + str(i) for i in range(1, len(muestra) + 1)]

plt.figure(figsize=(12, 6))
plt.bar(muestra["producto_n"], muestra["precio_num"])

plt.title("Precio de los primeros 20 productos con precio disponible")
plt.xlabel("Producto")
plt.ylabel("Precio")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)

plt.show()

### Interpretación del gráfico de barras de precios

El gráfico de barras permite comparar los precios de los primeros productos obtenidos.  
Gracias a esta visualización, se puede observar qué productos son más económicos y cuáles tienen un precio más elevado dentro de la muestra analizada.